# 📓 Semana 4 · Dia 3 — Otimização: OPTIMIZE, Liquid Clustering e VACUUM

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA, DEP (performance) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Tabela clusterizada + OPTIMIZE aplicado |

---


## 📖 Teoria — Por que otimizar?

A cada escrita, o Delta cria novos Parquet pequenos. Com o tempo: **muitos arquivos pequenos** → leituras lentas (muito overhead). O `OPTIMIZE` compacta arquivos pequenos em maiores (bin-packing).


## 📖 Teoria — Particionamento vs Liquid Clustering

**Particionamento** (`PARTITIONED BY (ano)`) divide em pastas fixas por coluna de baixa cardinalidade. Bom para data/hora; ruim para colunas de alta cardinalidade (muitas partições = muitos arquivos minúsculos).

**Liquid Clustering** (padrão 2026) usa `CLUSTER BY` e re-organiza dados de forma adaptativa (Z-ORDER foi deprecado — use CLUSTER BY):

```sql
CREATE TABLE t (col1 STRING, col2 INT) USING DELTA CLUSTER BY (col1, col2);
```

Vantagem: mantém o clustering automático em cada escrita e funciona com alta cardinalidade. **Regra**: mais de 1 TB → cluster por 1–4 colunas; abaixo disso, o OPTIMIZE ocasional é suficiente.


## 📖 Teoria — OPTIMIZE e VACUUM

`OPTIMIZE` reescreve e reordena arquivos; `OPTIMIZE ... ZORDER BY` ordenava por coluna (legado; com Liquid Clustering use `CLUSTER BY`). `VACUUM` remove arquivos órfãos e versões antigas (liberando espaço).


### 💻 Na prática — Criando tabela com Liquid Clustering

Crie a tabela de fatos já clusterizada — o padrão de produção.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.prata.fato_vendas_teste (
  InvoiceNo STRING, StockCode STRING, CustomerID STRING, Country STRING,
  data_venda DATE, quantidade INT, valor DOUBLE)
USING DELTA
CLUSTER BY (Country, data_venda);
SELECT COUNT(*) FROM workspace.prata.fato_vendas_teste;

In [ ]:
# Popular a tabela clusterizada (a partir do Bronze)
from pyspark.sql.functions import to_date
df = (spark.table("workspace.bronze.vendas_bronze")
    .select("InvoiceNo", "StockCode", "CustomerID", "Country",
            to_date("InvoiceDate", "M/d/yyyy H:mm").alias("data_venda"),
            "Quantity", "UnitPrice")
    .filter("CustomerID IS NOT NULL"))
df.write.mode("overwrite").saveAsTable("workspace.prata.fato_vendas_teste")
print("Populado:", spark.table("workspace.prata.fato_vendas_teste").count())

In [ ]:
# OPTIMIZE: compactar arquivos pequenos
spark.sql("OPTIMIZE workspace.prata.fato_vendas_teste")
print("Arquivos antes/depois (veja o Spark UI e o diretório da tabela).")

### 💻 Na prática — Analisando o efeito

Compare o número de arquivos antes e depois do OPTIMIZE.


In [ ]:
# Ver os arquivos físicos da tabela
display(spark.sql("DESCRIBE DETAIL workspace.prata.fato_vendas_teste").select("location"))
tabela = spark.table("workspace.prata.fato_vendas_teste")
print("NumFiles (do delta log):", spark.sql("DESCRIBE DETAIL workspace.prata.fato_vendas_teste").select("numFiles").collect()[0][0])

> 🎯 **Dica de prova**: DEA 2026 usa **Liquid Clustering** (CLUSTER BY) como recomendação; Z-ORDER é legado. Pergunta típica: particionamento vs clustering para coluna de alta cardinalidade → clustering.


## 🎯 Exercícios de fixação

**1.** Quando usar particionamento vs Liquid Clustering?

**2.** O que OPTIMIZE faz internamente?

**3.** Crie uma tabela com CLUSTER BY em 2 colunas e rode OPTIMIZE.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Particionar vs cluster

Particionar: colunas de baixa cardinalidade e consultas por filtro exato (ex.: ano). Cluster: alta cardinalidade, múltiplas colunas de filtro, atualizações frequentes.

**2.** OPTIMIZE

Compacta arquivos pequenos (bin-packing) e, com clustering, reordena dados — reduz overhead de leitura e acelera consultas por filtro.

**3.** Prática

`CREATE TABLE ... USING DELTA CLUSTER BY (col1, col2)` + `OPTIMIZE t`.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*